# Week 3 - Classification Lesson

In [1]:
## 3.1. PROJECT OVERVIEW - PREDICT CUSTOMER CHURN AT A TELCOM COMPANY

## 3.2. DATA PREPARATION

In [8]:
# 3.1. IMPORT PACKAGES
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

In [9]:
# 3.2 DOWNLOAD AND PREVIEW DATA
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data)
df.head()

# !wget $data -O data-week-3.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
# 3.3. STANDARDIZE COLUMN NAMES
# Preview column names
print(f"Column Names Before:\r\n {df.columns}")

# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

# Preview column names
print()
print(f"Column Names After:\r\n {df.columns}")

Column Names Before:
 Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

Column Names After:
 Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')


In [11]:
# 3.4 REVIEW pandas ASSIGNED COLUMN DATA TYPES
df.dtypes


customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges         object
churn                object
dtype: object

In [12]:
# 'seniorcitizen' COLUMN HOLDS ONLY 1 AND 0 VALUES AS IF THIS IS A TRUE/FALSE VALUE
# WE CAN TREAT THIS AS A CATEGORICAL VALUE LATER

# 'totalcharges' WAS MISTYPED AS A STRING WHEN IT CLEARLY CONTAINS NUMERIC VALUES
# WE SHOULD CONVERT THIS TO A NUMERIC FIELD

# USE THE to_numeric() METHOD OF THE pandas PACKAGE TO CONVERT 'totalcharges' COLUMN 
# FROM STRING OBJECT TO NUMERIC
pd.to_numeric(df.totalcharges)

#   NOTE: THIS SHOULD THROW AN ERROR

ValueError: Unable to parse string "_" at position 488

In [13]:
# THIS FAILED BECAUSE THERE WERE EMPTY SPACES IN THE RAW DATA IN THIS COLUMN
#  RECALL THAT WE REPLACED ALL SPACE CHARACTERS WITH UNDERSCORES, 
#  ERROR OCCURS BECAUSE 'pandas' IS UNABLE TO CONVERT AN UNDERSCORE TO A NUMBER

In [14]:
# CONVERT NUMERIC STRINGS TO NUMBERS - PART 1
#   STEP 1: USE THE 'errors='coerce' PARAMETER TO REPLACE CONVERSION ERRORS WITH 'NaN'
tc = pd.to_numeric(df.totalcharges, errors='coerce')

#   STEP 2: COUNT 'NaN' ERRORS
print(f"tc.isnull().sum()\t{tc.isnull().sum()}")

#   STEP 3: INSPECT 'NaN' records
#    df                              # Full dataframe
#    tc                              # dataframe of indices where the 'tc.isnull()' 
#                                    #  test returns a 'true'
#    ['customerid', 'totalcharges']  # List of columns to return from dataframe 'df'
df[tc.isnull()][['customerid', 'totalcharges']]

tc.isnull().sum()	11


,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


In [15]:
# CONVERT NUMERIC STRINGS TO NUMBERS - PART 2
#  SINCE 'totalcharges' REPRESENTS CUSTOMERS WHO JUST JOINED, 
#  THE INITIAL BALANCE SHOULD BE ZERO
#    WE SHOULD REPLACE 'NaN' VALUES WITH ZEROS
#    THIS MAKES SENSE BECAUSE NEW CUSTOMERS HAVE NOT PAID ANYTHING YET
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

In [16]:
# PREVIEW 'churn' CATEGORICAL COLUMN CONTENTS
df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: object

In [17]:
# CONVERT 'churn' CATEGORICAL VALUE
#  CONVERT 'yes' AND 'no' VALUES TO INTEGERS
#  df.churn column will be updated
#  where df.churn == 'yes', update to '1', otherwise update to '0'
df.churn = (df.churn == 'yes').astype(int)

## 3.3. VALIDATION

In [18]:
# IMPORT train-test-split FROM PACKAGE sklearn
from sklearn.model_selection import train_test_split

In [19]:
# SET ASIDE 20% FOR THE 'df_test' TEST DATASET
#   df_full_train    - name of dataset receive the bulk of the records taken from the full dataset
#   df_test          - name of remainder dataset to receive the remainder of the records from the full dataset
#   train_test_split - method within the 'model_selection' namespace of the 'sklearn' package
#   df               - dataframe containing all records
#   test_size        - percentage of full dataset to place into the remainder dataset
#   random_state     - seed for the random number generator
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)



In [20]:
# REPEAT THE PROCESS, BUT THIS TIME SPLIT THE 'df_full_train' INTO 'df_train' AND 'df_val'
#  SINCE WE WANT df_val TO CONTAIN 20% OF THE FULL 'df' DATASET, WE WILL NEED TO RECALCULATE
#  THE 'test_size' PARAMETER - 25% OF THE 'df_full_train' DATASET REPRESENTS 20% OF THE FULL 
#  'df' DATASET
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [21]:
# CONFIRM DATASET SIZES
print(f"length(df)             {len(df)}")
print(f"\tdf_train df_val\tdf_test")
print(f"\t{len(df_train)} \t {len(df_val)} \t {len(df_test)}")
print(f"Sum of split datasets: {len(df_train) + len(df_val) + len(df_test)}")


length(df)             7043
	df_train df_val	df_test
	4225 	 1409 	 1409
Sum of split datasets: 7043


In [22]:
# RESET INDEX VALUES FOR ALL SPLIT DATASETS
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

In [23]:
# EXTRACT TARGET / LABEL FROM EACH DATASET
y_train  = df_train.churn.values
y_val    = df_val.churn.values
y_test   = df_test.churn.values

In [24]:
# NOW WE DELETE THE 'churn' COLUMNS FROM FEATURE DATASETS
#  ** DO NOT DELETE BEFORE TARGET / LABEL VECTORS HAVE BEEN CREATED **
del df_train['churn']
del df_val['churn']
del df_test['churn']

## 3.4. EDA

In [26]:
# reset index
df_full_train = df_full_train.reset_index(drop=True)

In [27]:
# Check all columns for missing values
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [29]:
# look at the target variable - how many customers churned and how many stayed
df_full_train.churn.value_counts(normalize=True)

print("About 73% of the customers stayed and 27% churned")

About 73% of the customers stayed and 27% churned


In [30]:
# Since 'churn' holds only binary data there is a shortcut for calculating the churn rate
#  We can use the .mean() function
#  with binary data, the function performs a sum of all values (1s and 0s), and divides by the total number of rows
df_full_train.churn.mean()

np.float64(0.26996805111821087)

In [32]:
# Decide which columns are hold categotical data
#  This can be done with a visual inspection of the data

# Create a list of columns containing numeric values
numerical = [
    'tenure', 
    'monthlycharges', 
    'totalcharges'
] 

# Create a list of columns containing categorical values
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

# check number of unique values in each categorical column
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

## 3.5. Risk

In [34]:
# Determining relevant features is part of EDA

# PART 1: Reference point
global_churn = df_full_train.churn.mean()

In [38]:
# PART 2: Review gender impact upon churn
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_male   = df_full_train[df_full_train.gender == 'male'].churn.mean()

print(f"churn_female: {churn_female}")
print(f"churn_male:   {churn_male}")

churn_female: 0.27682403433476394
churn_male:   0.2632135306553911


In [39]:
# Check to see how much of the global churn rate can be explained by female gender
global_churn - churn_female

np.float64(-0.006855983216553063)

In [40]:
# Check to see how much of the global churn rate can be explained by male gender
global_churn - churn_male

np.float64(0.006754520462819769)

In [ ]:
# INTERPRETATION: These factors have almost zero impact on global_churn, they are likely not important features

In [41]:
# PART 3: Review whether a customer has a partner impacts churn
df_full_train.partner.value_counts()

partner
no     2932
yes    2702
Name: count, dtype: int64

In [42]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

print(f"churn_partner:    {churn_partner}")
print(f"churn_no_partner: {churn_no_partner}")

churn_partner:    0.20503330866025166
churn_no_partner: 0.3298090040927694


In [43]:
# # Check to see how much of the global churn rate can be explained by customer having a partner
global_churn - churn_partner     #  0.06493474245795922

np.float64(0.06493474245795922)

In [44]:
# # Check to see how much of the global churn rate can be explained by customer not having a partner
global_churn - churn_no_partner  # -0.05984095297455855

np.float64(-0.05984095297455855)

In [45]:
# INTERPRETATION: 
#    Positive rate churns less than average
#    Negative rate churns more than average
#    The larger the absolute distance, the more significant the feature

In [ ]:
# RISK RATIO
#
#    Risk_Ratio = Group_Risk / Global_Risk

In [47]:
from IPython.display import display

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count']) # Group by 'c' in 'categorical' list
                                                                     # then create new 'mean' and 'count' aggregate feature columns for 'churn'
    df_group['diff'] = df_group['mean'] - global_churn               # Add a 'diff' column
    df_group['risk'] = df_group['mean'] / global_churn               # Add a 'risk' column
    display(df_group)                                                # 'IPython' package method to display a table for each iteration of a loop
    print()
    print()
    

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


print("INTERPRETATION")   \
print("  1. customers on a month-to-month contract churn far more than average, while two-year contracts almost never churn")   \
print("  2. fiber optic customers churn much more than DSL customers")   \
print("  3. customers paying by electronic check are the riskiest ")   \
print("")   \
print("CONCLUSION:")   \
print("  internet service and payment method look important; gender and phone service do not")   \

Notes   \
   \
Churn rate: Difference between global mean of the target variable and mean of the target variable for categories of a feature. If this difference is greater than 0, it means that the category is less likely to churn, and if the difference is lower than 0, the group is more likely to churn. The larger differences are indicators that a variable is more important than others.   \

Risk ratio: Ratio between mean of the target variable for categories of a feature and global mean of the target variable. If this ratio is greater than 1, the category is more likely to churn, and if the ratio is lower than 1, the category is less likely to churn. It expresses the feature importance in relative terms.   \
   \
Functions and methods:   \
   \
df.groupby('x').y.agg([mean()]) - returns a dataframe with mean of y series grouped by x series   \
display(x) displays an output in the cell of a jupyter notebook.   \

## MUTUAL THEORY (USE WITH CATEGORICAL DATA)  \
Mutual information is a concept from information theory. It tells us how much we can learn about one variable if we know the value of another.   \
   \
In concept, this means that if we have a Venn diagram with two partially overlapping circles and we understand one circle, then we know something about the elements of the second circle when they overlap with the first circle.  Therefore, understanding how much of the second circle overlaps with the first circle is important.  \


In [50]:
# Package 'sklearn' includes a function to calculate a mutual information score
from sklearn.metrics import mutual_info_score

In [54]:
# Measure the score for a the 'contract' feature column
#   Order of arguments does not matter
mutual_info_score(df_full_train.churn, df_full_train.contract)

0.0983203874041556

In [55]:
# Measure the score for a the 'gender' feature column
#   Order of arguments does not matter
mutual_info_score(df_full_train.gender, df_full_train.churn)

0.0001174846211139946

In [56]:
# Measure the score for a the 'partner' feature column
#   Order of arguments does not matter
mutual_info_score(df_full_train.partner, df_full_train.churn)

0.009967689095399745

### INTERPRETATION:   \
  1. The numbers confirm what we saw with the risk ratios.    \
  2. Contract has by far the highest score: knowing the contract type teaches us a lot about churn.    \
  3. Gender is almost 0: knowing the gender tells us practically nothing.    \
  4. Partner sits in between.   


In [57]:
## BETTER CODING APPROACH
#   This method automatically calculates the mutual info score for each categorical column/feature

# Declare a mutual_info_churn function
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

# Apply 'mutual_info_churn_score' function to each column in the 'categorical' list
#   NOTE: '.apply() is a 'pandas' package function
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

### Notes\
   \
Mutual information is a concept from information theory, which measures how much we can learn about one variable if we know the value of another. In this project, we can think of this as how much do we learn about churn if we have the information from a particular feature. So, it is a measure of the importance of a categorical variable.\
\
Classes, functions, and methods:\
\
mutual_info_score(x, y) - Scikit-Learn class for calculating the mutual information between the x target variable and y feature.\
df[x].apply(y) - apply a y function to the x series of the df dataframe.\
 df.sort_values(ascending=False).to_frame(name='x') - sort values in an ascending order and called the column as x.\



## 3.7. CORRELATION: (USE WITH NUMERIC VALUES)\
The **correlation coefficient** - the Pearson correlation coefficient, usually denoted r - measures the degree of dependency between two variables. It is always a number between -1 and 1:\
\
A value __close to 1 means strong positive correlation__:  when one variable grows, the other grows too.\
A value __close to -1 means strong negative correlation__: when one variable grows, the other decreases.\
A value __close to 0 means there is no dependency__:       a change in one variable tells us nothing about the other.\
The __sign gives the direction of the relationship__, and the _absolute value gives its strength_.\

One way to interpret the magnitude:\
\
**LOW** when the absolute value of r is between **0 and 0.2**\
**MEDIUM** when it is **between 0.2 and 0.5**\
**STRONG** when it is **between 0.5 and 1.0**\
\
If the correlation between a numerical feature and churn is positive, customers with **higher values of that feature churn more**; if it is negative, they churn less. The **larger the absolute value, the more important the feature**.\


In [60]:
# COMPUTE CORRELATION
#    'corrwith' is a 'pandas' function
#    'numerical' is a list of numerical columns/features
df_full_train[numerical].corrwith(df_full_train.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

### INTERPRETATION: \
- **Tenure** has a medium negative correlation. The longer customers stay with the company, the less likely they are to churn. \
- **Monthly charges** have a low-to-medium positive correlation: the more customers pay per month, the more likely they are to churn. \
- **Total charges** have a low-to-medium negative correlation - not surprising, since total charges accumulate with tenure. \

In [61]:
# To rank importance of features, consider the absolute value of the difference
df_full_train[numerical].corrwith(df_full_train.churn).abs()

tenure            0.351885
monthlycharges    0.196805
totalcharges      0.196353
dtype: float64

### INTERPRETATION \
Tenure is the most important numerical feature.

In [64]:
# calculate 'tenure' value range (measured in months)
print(df_full_train.tenure.min())
print(df_full_train.tenure.max())

0
72


In [73]:
# break 'tenure' into 3 groups
# calculate mean for each group
# groups: 
#   1. less than 2 months
#   2. up to a year
#   3. more than a year

print(f"                x  <   2 months : {df_full_train[df_full_train.tenure <= 2].churn.mean()}")
print(f"  2 months  <=  x  <  12 months : {df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean()}")
print(f" 12 months  <=  x               : {df_full_train[(df_full_train.tenure >= 12)].churn.mean()}")

                x  <   2 months : 0.5953420669577875
  2 months  <=  x  <  12 months : 0.3994413407821229
 12 months  <=  x               : 0.17986881937436933


### INTERPRETATION
The pattern is striking. Customers who have been with the company for two months or less churn at almost 60%. \
For customers of up to a year it is about 40%, and for \
those who stayed longer than a year only about 18%. \
\
Churn clearly decreases as tenure grows - exactly what the negative correlation told us.

In [72]:
# Perform same check for 'monthlycharges'
print(f"         x  <  $20 : {df_full_train[df_full_train.monthlycharges <= 20].churn.mean()}")
print(f" $20  <= x  <  $50 : {df_full_train[(df_full_train.monthlycharges > 20) & (df_full_train.monthlycharges <= 50)].churn.mean()}")
print(f" $50  <= x         : {df_full_train[(df_full_train.monthlycharges >= 50)].churn.mean()}")


         x  <  $20 : 0.08795411089866156
 $20  <= x  <  $50 : 0.18340943683409436
 $50  <= x         : 0.3248223216635957


### INTERPRETATION
Here the relationship goes the other way: customers paying at most $20 churn at about 9%,   \
and customers paying more than $50 churn at about 32%. \
The more people pay monthly, the more likely they are to leave - the positive correlation confirmed. \

## Notes 
\
Correlation coefficient measures the degree of dependency between two variables. This value is negative if one variable grows while the other decreases, and it is positive if both variables increase. Depending on its size, the dependency between both variables could be low, moderate, or strong. It allows measuring the importance of numerical variables. \
\
If r is correlation coefficient, then the correlation between two variables is: \
\
**LOW** when r is between [0, -0.2) or [0, 0.2) \
**MEDIUM** when r is between [-0.2, -0.5) or [0.2, 0.5) \
**STRONG** when r is between [-0.5, -1.0] or [0.5, 1.0] \
Positive Correlation vs. Negative Correlation \
\
When r is positive**, an increase in x will increase y. \
When r is negative**, an increase in x will decrease y. \
When r is 0, a change in x does not affect y. \
Functions and methods: \
\
df[x].corrwith(y) - returns the correlation between x and y series. This is a function from pandas. \


## OHE (ONE-HOT ENCODING)
One-hot encoding replaces one categorical column with several binary columns - one per category.

### 3.8. OHE (ONE-HOT ENCODING)
One-hot encoding replaces one categorical column with several binary columns - one per category. \
Only one of the created binary columns will store a '1' and all the others will store a '0' for each row \
The column with the '1' is called the 'hot' column \
 \
 'sklearn' has a function that automates the work of transforming a categorical feature into a collecton of one-hot columns.  This function requires the input data to be in a dictionary structure.

In [81]:
# CREATE THE TRAINING DATASET

# Import function used to apply one-hot encoding (OHE)
from sklearn.feature_extraction import DictVectorizer

# Create the 'dv' matrix 
dv = DictVectorizer(sparse=False) 

# Create the training dictionary
train_dict = df_train[categorical + numerical].to_dict(orient='records')

# Create training dataset with one0hot encoded columns
X_train = dv.fit_transform(train_dict)

In [75]:
# Create dictionary
#   orient='records' means that there will be one dictionary entry created for each dataframe row
dicts = df[categorical + numerical].to_dict(orient='records')

In [77]:
# BONUS: Preview 'dicts'
# dicts

In [78]:
# Like every Scikit-Learn transformer, it has a fit step and a transform step:
# 
#    'fit'            learns the list of categories for each column
#    'transform'      turns dictionaries into a matrix
#    'fit_transform'  does both in one call. 
# With 
#    'sparse=False'   the result is a usual NumPy array. 
#
# By default 'DictVectorizer' returns a sparse matrix 
#
# a SPARSE MATRIX is a data structure that stores only the non-zero elements, 
#   because most of the matrix is zeros. Sparse matrices are more memory efficient, 
#   but for this dataset it is simpler to work with a dense array.

In [82]:
# ENCODE THE VALIDATION DATASET

# Import function used to apply one-hot encoding (OHE)
from sklearn.feature_extraction import DictVectorizer

# Create the 'dv' matrix 
dv = DictVectorizer(sparse=False) 

# Create the training dictionary
train_dict = df_train[categorical + numerical].to_dict(orient='records')

# Create training dataset with one0hot encoded columns
X_train = dv.fit_transform(train_dict)

# preview training dataset
X_train.shape

(4225, 45)

In [85]:
# ENCODE THE VALIDATION DATASET
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

### Notes
\
One-Hot Encoding allows encoding categorical variables in numerical ones. This method represents each category of a variable as one column, and a 1 is assigned if the value belongs to the category or 0 otherwise. \
\
Classes, functions, and methods: \
\
df[x].to_dict(orient='records') - convert x series to dictionaries, oriented by rows. \
DictVectorizer().fit_transform(x) - Scikit-Learn class for one-hot encoding by converting x dictionaries into a sparse matrix. It does not affect the numerical variables. \
DictVectorizer().get_feature_names() - return the names of the columns in the sparse matrix. \